In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.1 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from datasets import load_dataset
import numpy as np
import random

In [ ]:
# set seed value for reproduce
seed_value = 42
random.seed(seed_value)
torch.manual_seed(seed_value)
np.random.seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

Load dataset- data preprocessing

In [ ]:
#load dataset
#imdb for sentiment analysis: 0= neg, 1=post
dataset= load_dataset('imdb')

max_length= 256 #max length of one sequence (calc every review in corpus then plot dist and then see what is optimal)
#may reduce or increase this depending on your resources (some reviews may be short-> so longer than the length will be truncated or if shorter then length will be padded to length [PAE] is padding token inputted in BERT)
train_texts= dataset['train']['text']
train_labels= dataset['train']['label']
test_texts= dataset['test']['text'] #use for test split for validation
test_labels= dataset['test']['label']

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(train_texts[0])
print(train_labels[:10]) #first 10
print(train_labels[-10:]) #see labels for last 10 reviews

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

Initialize bert tokernizer


In [ ]:
#initialize the bert tokensizer and tokenize the data
tokenizer= BertTokenizer.from_pretrained('bert-base-uncased')
#uncased- all in lovwercase?
#want every review to be seq of subword tokens
train_encodings= tokenizer(train_texts, truncation= True, padding= True, max_length= max_length)
test_encodings= tokenizer(test_texts, truncation= True, padding= True, max_length= max_length)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
#want them as ids rather than strings, for every review want a list of integers
#truncated- if review after bpe is more than max_length then truncate
# if less than max length then we will pad
#keys in dict
print(train_encodings.keys())
#10,000 X 256 if 10,000 is training size

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:

# dict with 3 elements:
#101= cls
#102= sep

print(train_encodings['input_ids'][0])
print(train_encodings['attention_mask'][20]) #21st review (21st row): everything is 1 bcs encoder
print(train_encodings['token_type_ids'][0]) #

[101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107, 2004, 1996, 5148, 2162, 1998, 2679, 3314, 1999, 1996, 2142, 2163, 1012, 1999, 2090, 4851, 8801, 1998, 6623, 7939, 4697, 3619, 1997, 8947, 2055, 2037, 10740, 2006, 4331, 1010, 2016, 2038, 3348, 2007, 2014, 3689, 3836, 1010, 19846

In [ ]:
decoded_token= tokenizer.decode(101)
print(decoded_token)
#decode a token in vocab

[CLS]


In [ ]:
#get number of unique tokens (from input_ids bcs they are unique)
unique_tokens = set()
for encoding in train_encodings['input_ids']:
  unique_tokens.update(encoding)

print(f"Number of unique tokens in training set: {len(unique_tokens)}")

Number of unique tokens in training set: 24489


define a pytorch dataset wrapper

In [ ]:
# define a pytorch dataset wrapper to handle tokenized inputs and labels


#blue print is the class- based on that can make many objects
# inside class define the functions and methods--> so when create object (from class) what attributes and functions etc
class SentimentDataset(Dataset):
  def __init__(self, encodings, labels): #real sentiment needs encoding and lables for initialization
    self.encodings= encodings
    self.labels= labels

  def __getitem__(self, idx):
    #convert lists to tensors
    item= {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} #use a for loop to iterate through dict and get key and value (seq of ___--> then convert to tensor)---> 3 iterations: input ids, attemtion masks, token types id and get key and value from those
    #list off tuples with key, value
    item['labels']= torch.tensor(self.labels[idx]) #key= label and get coresponding label id for that review
    return item

  def __len__(self):
    return len(self.labels)

create dataset and datalaoder

In [ ]:
#dataset objects created--> encoding and label
train_dataset= SentimentDataset(train_encodings, train_labels)
test_dataset= SentimentDataset(test_encodings, test_labels)

In [ ]:
train_dataset[0]

{'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
         25430, 14728,  2245,  2055,  3

In [ ]:
#create dataloaders for train and eval
#iterable object --> for dataloader
train_loader= DataLoader(train_dataset, batch_size= 8, shuffle= True) #shuffle- so for new epoch there is a shuffle
test_loader= DataLoader(test_dataset, batch_size= 8, shuffle= False) #only one epoch with testing instances

In [ ]:
# first instance in that batch
next(iter(train_loader))['input_ids'][0]

tensor([  101,  1045,  2293,  9752, 17323,  1012,  1998,  2008,  1005,  1055,
         3038,  1037,  2843,  2044,  2023,  2143,  1012,  2941,  1010,  2002,
         2003,  2025,  2919,  1999,  2009,  1012,  2002,  2074,  3849,  2000,
         3233,  4998,  1010,  2022,  3923,  2063,  1998,  2010,  5156, 26380,
         2969,  1010,  2021, 15697, 23233,  2050,  1012,  2009,  2003,  5793,
         1996,  2611,  2002,  2003, 10349,  2007,  2003,  1037, 27145,  1010,
         2130,  2004,  2019, 26252,  2402,  2413,  2611,  1012,  2909,  9752,
         2876,  1005,  1056,  2031,  4217,  2014,  2043,  2002,  2001,  2402,
         1998,  2200,  5525,  3475,  1005,  1056,  2205,  3407,  2055,  2009,
         2085,  1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,
         1996,  5875,  2839,  2003,  1996, 28902,  2567,  1997,  1996,  5976,
         1000, 15146,  1000,  1010,  2178,  1056,  9148,  2102,  1012,  1000,
         6221,  1000,  2004, 20781,  2015,  2000,  2022,  1037, 

In [ ]:
next(iter(train_loader)) #gives full batch
#0 in attention- truncation and padding
#8 rows for batch and 256 columns for size

{'input_ids': tensor([[  101,  1045,  2179,  ...,  1997,  2107,   102],
         [  101, 10556, 28029,  ...,  2003,  9033,   102],
         [  101,  4869,  8716,  ...,     0,     0,     0],
         ...,
         [  101,  2034,  1997,  ...,     0,     0,     0],
         [  101,  2009,  1005,  ..., 20193,  2668,   102],
         [  101,  1045,  2074,  ...,     0,     0,     0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'labels': tensor([1, 0, 1, 1, 1, 0, 1, 0])}

Training Prep

In [ ]:
# initialize the pretrained bert model for sequence classification
model=BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2) #labels for classification

#move model to GPU if avilable
device= torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

#set optimizer and learning rate scheduler
optimizer= AdamW(model.parameters(), lr= 2e-5) #full fine tuning with all model parameters
#2e-5= learning rate #aaptive learning rate with momentum w= adaptive weight decay--? earlier training= larger weight then decrease
epochs= 3
total_steps= len(train_loader)* epochs #how many iterations = num of batches* num of epochs

#optimizer: This is your Pyorch optimizer (AdamW) that will be used to update the models parameters. Scheduler adjusts the LR within this optimizer
#num_warmup_steps: This argument specifies the number of traiing steps during the which the LR will be gradually increased from a starting value (usuallu 0) to the base LR you set in your...
#num_training_steps: This is the total number of training steps you plan to run

scheduler= get_linear_schedule_with_warmup(optimizer,
                                           num_warmup_steps= 0, #not using a warm-up phase
                                           num_training_steps= total_steps)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


training porcess

In [ ]:
#training loop


# for eahc run 8 labels and 8 seq for attention mask

model.train() #set model to training mode
for epoch in range(epochs):
  total_loss= 0
  for batch in train_loader:
    #zero the gradients
    optimizer.zero_grad() #reset grad for every iteration

    #move batch to device
    input_ids= batch['input_ids'].to(device)
    attention_mask= batch['attention_mask'].to(device)
    labels= batch['labels'].to(device)

    #forward pass (BERT returns loss when labels are provided)
    outputs= model(input_ids, attention_mask= attention_mask, labels= labels) # provide seq of ids, attention mask, labels for training--> later in inference do not need
    loss= outputs.loss
    total_loss+= loss.item() #loss value (item= GPU)

    #backward pass and optimization-gradient descent
    loss.backward()
    optimizer.step()
    scheduler.step()

  #print average loss per epoch
  avg_train_loss= total_loss/ len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}")


Epoch 1/3, Train Loss: 0.2469


KeyboardInterrupt: 

In [ ]:
print(model.summary()) #bert base 12
# word embed= token embed
#pooler= classofier
#dense= FFN
#activation = softmax

Evaluation

In [ ]:
# evaluate on test set--> only test so 1 epoch
model.eval()
correct=0
total=0

with torch.no_grad():
  for batch in test_loader:
    #move batch to device
    input_ids= batch['input_ids'].to(device)
    attention_mask= batch['attention_mask'].to(device)
    labels= batch['labels'].to(device)

    #forward pass
    outputs= model(input_ids, attention_mask= attention_mask)
    logits= outputs.logits #raw 8x2 (prob of 0 and prob of 1)

    # get predicitons from logits
    predictions = torch.argmax(logits, dim=-1) #index with largest value across last dimension (column=1)--> for every row look at max column--> 8x1
    correct += (predictions == labels).sum.item() #labels is also 8x1--> elementalwise comparsion, returns boolean value (T/F) and can be treated as 0/1 then sum it add = how many correct class in batch
    total += labels.size(0) #labels= 8x 1 so size(0) gives how many instances in batch aka 8

accuracy= correct/total
print(f"Test Accuracy: {accuracy:.4f}")